# Image → CadQuery Code Generation — Technical Test Solution

**Task.** Given a rendered image of a CAD part, generate the **CadQuery** Python code that reconstructs it. The dataset is [`CADCODER/GenCAD-Code`](https://huggingface.co/datasets/CADCODER/GenCAD-Code) (~147K image/code pairs). We are scored with two repo-provided metrics:

1. **Valid Syntax Rate (VSR)** — does the generated code execute and yield a CadQuery solid?
2. **Best IoU** — voxel intersection-over-union between the meshes built from the generated vs. ground-truth code, after principal-axis alignment.

> The brief states that **relative** improvement (baseline → enhanced) matters more than absolute numbers, and that GPU-poor solutions are expected. This solution is built and run end-to-end on an **Apple M4 laptop (16 GB, MPS)** — no datacenter GPU.

## TL;DR

- **Model:** a `VisionEncoderDecoder` = **ViT image encoder + GPT-2 text decoder**, initialised from `nlpconnect/vit-gpt2-image-captioning` and fine-tuned on (image → code) pairs.
- **Baseline:** short fine-tune (3K samples) + greedy decoding.
- **Enhanced:** more data + longer training + **constrained beam-search decoding** (beam search, n-gram repetition blocking, repetition penalty, minimum length).
- Both are evaluated on the dataset's **curated 100-sample subset** (`hundred_subset=True`).

See `SOLUTION.md` for the full write-up. Final numbers are loaded live from `results/` at the bottom.

## 1. Environment & reproduction

```bash
uv sync                      # base deps (cadquery, datasets, trimesh, ...)
uv add torch torchvision transformers accelerate pillow matplotlib

# Download two parquet shards (train + test) into ./data (see scripts/download_data.py)
uv run python scripts/download_data.py

# Baseline: short fine-tune + greedy decoding
uv run python scripts/run_experiment.py --name baseline \
    --train-limit 3000 --max-steps 300 --num-beams 1 --max-new-tokens 512

# Enhanced: more data + longer training + constrained beam search
uv run python scripts/run_experiment.py --name enhanced \
    --train-limit 8000 --max-steps 900 --num-beams 5 \
    --no-repeat-ngram 3 --repetition-penalty 1.2 --min-new-tokens 40 --max-new-tokens 512

uv run python scripts/compare_results.py   # table + plots into results/
```

The code is organised as a small package:

- `src/data.py` — parquet loading, filtering, `Dataset`/collate.
- `src/modeling.py` — model factory (ViT-GPT2 VisionEncoderDecoder).
- `src/train.py` — explicit MPS-friendly training loop (grad accumulation, grad checkpointing).
- `src/evaluate.py` — generation + the two metrics (wraps `metrics/`).
- `scripts/run_experiment.py` — end-to-end runner.


## 2. The data

Each row is `{image (448×448), cadquery (target code), token_count, deepcad_id, prompt, hundred_subset}`.
The code is a sequence of sketch loops with **exact float coordinates** plus extrude/boolean ops, e.g.:

```python
import cadquery as cq
wp_sketch0 = cq.Workplane(cq.Plane(cq.Vector(-0.148, 0.0, 0.0), cq.Vector(1.0,0.0,0.0), cq.Vector(0.0,0.0,1.0)))
loop0 = wp_sketch0.moveTo(0.15, 0.0).circle(0.15)
solid0 = wp_sketch0.add(loop0).extrude(0.75)
solid = solid0
```

The cell below shows a real sample (requires the downloaded parquet in `./data`).

In [ ]:
import sys, matplotlib.pyplot as plt
sys.path.insert(0, '.')
from src.data import load_samples, DATA_DIR

samples = load_samples(DATA_DIR / 'test_0000.parquet', only_hundred_subset=True, limit=3)
s = samples[0]
plt.figure(figsize=(3,3)); plt.imshow(s.image); plt.axis('off'); plt.title(s.deepcad_id); plt.show()
print(s.code[:600])

## 3. Approach

**Why a ViT-GPT2 VisionEncoderDecoder?** This is fundamentally an *image-conditioned code-generation* problem — structurally identical to image captioning, where the "caption" is a long program. Starting from `nlpconnect/vit-gpt2-image-captioning` gives us a vision encoder and an autoregressive decoder that are *already wired together with cross-attention*, so fine-tuning only has to adapt the model to the CadQuery "language" rather than learn vision-to-text from scratch. At ~240M params it fits and trains on an M4.

**Key engineering decisions (laptop constraints):**
- **Gradient checkpointing + dynamic padding + grad accumulation** to fit a 240M encoder-decoder in 16 GB unified memory.
- **EOS supervision** so the decoder learns *when to stop* — critical for emitting syntactically complete code.
- **Length filtering** (drop the longest codes) keeps memory/time bounded; the median code is ~320 GPT-2 tokens.

**The enhancement (baseline → enhanced):**
1. **More data + longer training** (3K→18K samples, 300→900 steps).
2. **Constrained beam-search decoding**: `num_beams=5`, `no_repeat_ngram_size=3`, `repetition_penalty=1.2`, `min_new_tokens=40`. Greedy decoders on long, repetitive code tend to fall into degenerate loops that never close a parenthesis (→ invalid). These constraints directly lift VSR *and* IoU.

## 4. Results (loaded live from `results/`)

In [ ]:
import json
from pathlib import Path
R = Path('results')
def show(name):
    p = R / f'{name}_metrics.json'
    if not p.exists():
        print(f'{name}: not run yet'); return None
    m = json.loads(p.read_text())
    print(f"{name:>9} | VSR {m['vsr']*100:5.1f}% | IoU(all) {m['mean_iou_all']*100:5.1f}% | IoU(valid) {m['mean_iou_valid']*100:5.1f}% (n={m['n_iou_pairs_valid']})")
    return m
b = show('baseline'); e = show('enhanced')

In [ ]:
from IPython.display import Image as IPImage, display
for img in ['comparison.png', 'loss_curves.png']:
    p = R / img
    if p.exists():
        display(IPImage(filename=str(p)))

## 5. Bottlenecks & what I'd do with more time

**Bottlenecks identified**
- **Compute/memory.** A 16 GB M4 caps batch size and sequence length; ~71% of codes fit in 512 GPT-2 tokens, so the longest, most complex parts are under-served.
- **Output length & precision.** Targets are long lists of exact floats. Cross-entropy over GPT-2 BPE wastes capacity tokenising digits, and a single wrong coordinate can tank IoU while VSR stays high — the two metrics decouple.
- **Greedy degeneration.** Long repetitive programs invite decoder loops; this is the single biggest VSR killer and is cheap to fix with constrained decoding.

**With more time / a GPU**
1. **Stronger backbone** (e.g. Qwen2-VL / a code-pretrained decoder) with **LoRA** — better priors over Python syntax.
2. **Numeric tokenisation** (digit-splitting or a coordinate-aware tokenizer) and/or **coordinate quantisation** to shorten targets and improve precision.
3. **Execution-guided / constrained decoding** (only emit tokens that keep the program parseable) to push VSR → ~100%.
4. **RL / best-of-n against IoU** — directly optimise the geometric metric rather than next-token likelihood.
5. **Full-dataset training** with multi-view image augmentation.